# Problem Set 3 - Parallel Implementation

## Setup

This notebook tackles the same problem as `PS3_new.ipynb` but leverages parallel processing to accelerate the Monte Carlo simulations. The core logic remains the same, but the execution is distributed across multiple CPU cores.

In [ ]:
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed
import time

## Question 1-3: Simulation and Bootstrap CI Function

We define a single function, `run_single_simulation`, which encapsulates one entire cycle of the experiment for a given sample size `n`. This includes:
1. Generating the heteroskedastic data.
2. Computing the initial OLS estimate and robust standard error.
3. Running the bootstrap procedure (B=1000 times) to find the quantiles of the t-statistic distribution.
4. Constructing the 95% confidence interval.
5. Checking if the true value of β1 (which is 1) is contained within the interval.

This function is designed to be called in parallel.

In [ ]:
def run_single_simulation(n, B=1000):
    """
    Runs a single simulation for a given sample size n.
    Returns 1 if the confidence interval contains the true beta1, 0 otherwise.
    """
    # Constants
    B0 = 1
    B1 = 1

    # 1. Generate Data
    X = np.random.normal(1, 1, n)
    U = np.array([np.random.normal(0, xi**2) for xi in X])
    Y = B0 + B1 * X + U
    X_with_const = sm.add_constant(X)

    # 2. Compute OLS estimator and robust standard error
    ols_model = sm.OLS(Y, X_with_const).fit()
    beta1_hat = ols_model.params[1]
    # Use HC1 for robust standard errors as it's a common choice
    robust_se = ols_model.get_robustcov_results(cov_type='HC1').bse[1]

    # Bootstrap procedure to find quantiles
    t_stats = []
    for _ in range(B):
        # Draw a random bootstrap sample (with replacement)
        indices = np.random.choice(n, n, replace=True)
        X_b = X[indices]
        Y_b = Y[indices]
        X_b_with_const = sm.add_constant(X_b)
        
        # Compute OLS on bootstrap sample
        ols_model_b = sm.OLS(Y_b, X_b_with_const).fit()
        beta1_hat_b = ols_model_b.params[1]
        robust_se_b = ols_model_b.get_robustcov_results(cov_type='HC1').bse[1]
        
        if robust_se_b > 0:
            tb = (beta1_hat_b - beta1_hat) / robust_se_b
            t_stats.append(tb)
    
    if not t_stats:
        return 0 # Return 0 if bootstrap fails

    # Find the 0.025 and 0.975 quantiles
    q_025 = np.quantile(t_stats, 0.025)
    q_975 = np.quantile(t_stats, 0.975)

    # Construct the confidence interval
    ci_lower = beta1_hat - q_975 * robust_se
    ci_upper = beta1_hat - q_025 * robust_se

    # 3. Check if the interval contains the true beta1
    return 1 if ci_lower <= B1 <= ci_upper else 0

## Question 4 & 5: Running the Full Experiment in Parallel

Now we run the main experiment. We loop through each sample size `n` and, for each `n`, we execute the `run_single_simulation` function 1000 times in parallel.

`n_jobs=-1` tells `joblib` to use all available CPU cores. We record the total time taken for each full Monte Carlo run to demonstrate the performance benefits.

In [ ]:
N_VALUES = [10, 50, 100, 500, 1000]
NUM_SIMULATIONS = 1000
coverage_results = {}

print(f"Starting simulations... (Monte Carlo repetitions = {NUM_SIMULATIONS})\n")

for n in N_VALUES:
    start_time = time.time()
    
    # Use joblib to run the simulations in parallel
    # n_jobs=-1 uses all available CPU cores
    results = Parallel(n_jobs=-1)(delayed(run_single_simulation)(n) for _ in range(NUM_SIMULATIONS))
    
    # Calculate coverage
    coverage = np.sum(results) / NUM_SIMULATIONS
    coverage_results[n] = coverage
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    print(f"Results for n = {n}:")
    print(f"  Coverage probability: {coverage:.4f}")
    print(f"  Time taken: {elapsed_time:.2f} seconds\n")


## Analysis and Conclusion

In [ ]:
print("Summary of Coverage Probabilities:")
for n, coverage in coverage_results.items():
    print(f"  n = {n:<4}: {coverage:.4f}")

### How is it different from Problem 1 in Problem Set 1?

The key difference in this problem is the presence of **heteroskedasticity**, where the variance of the error term `U` depends on the regressor `X` (specifically, `Var(Ui|Xi) = Xi^2`). In Problem Set 1, the error term was likely homoskedastic (constant variance).

1.  **Need for Robust Standard Errors**: With homoskedasticity, standard OLS standard errors are valid. Here, they are not. That is why we must compute **robust standard errors** (e.g., HC1) and use them in our bootstrap procedure. The bootstrap procedure itself is designed to be robust to issues like heteroskedasticity.

2.  **Coverage Performance**: The results show how the bootstrap confidence interval's coverage performs at different sample sizes. We typically observe that for very small `n` (like n=10), the actual coverage probability may be lower than the nominal 95% level. This is because the asymptotic approximations that justify the bootstrap method work better with larger samples. As `n` increases (50, 100, 500, 1000), we expect the coverage probability to get closer and closer to the target of 0.95. This demonstrates that the bootstrap percentile-t method with robust standard errors is asymptotically valid and provides reliable confidence intervals for large enough samples, even under heteroskedasticity.